# Step 2: Splits, preprocessing, DVC
Re-run the download cells from notebook 01 first if this is a new Colab session (data lives in /content/rsna). Set GITHUB_URL below after you push the repo.

In [ ]:
!pip -q install --upgrade pydicom monai dvc

In [ ]:
# Get the project code (push the local repo to GitHub first)
GITHUB_URL = 'https://github.com/YOUR_USER/medai.git'  # <-- change this
!git clone {GITHUB_URL} /content/medai
%cd /content/medai

In [ ]:
from ml.dataset import build_splits
build_splits('/content/rsna/stage_2_train_labels.csv', 'data/splits')

In [ ]:
import pandas as pd
from ml.dataset import preprocess_all
ids = pd.concat([pd.read_csv(f'data/splits/{s}.csv') for s in ['train','val','test']]).patientId
preprocess_all('/content/rsna/stage_2_train_images', ids, 'data/processed')  # about 5-10 min
!ls data/processed | wc -l

In [ ]:
# Check the augmentation visually
import matplotlib.pyplot as plt
from ml.dataset import CXRDataset
ds = CXRDataset('data/splits/train.csv', 'data/processed', train=True)
fig, axs = plt.subplots(1, 4, figsize=(14,4))
for ax in axs:
    x, y = ds[0]; ax.imshow(x[0], cmap='gray'); ax.set_title(f'label {int(y)}'); ax.axis('off')
plt.show()  # same image, four different random augmentations

## DVC: version the data
DVC stores a tiny .dvc pointer file in git, while the real files go to a remote (here Google Drive mounted in Colab).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!dvc init
!dvc add data/processed
!dvc remote add -d gdrive /content/drive/MyDrive/medai_dvc
!dvc push
!git status --short

In [ ]:
# Commit pointers (NOT the images)
!git config user.email you@example.com
!git config user.name YourName
!git add data/processed.dvc data/.gitignore data/splits .dvc .dvcignore
!git commit -m 'Step 2: splits, preprocessed data tracked with DVC'
# then git push (needs a GitHub personal access token)